# Análisis Comercial Inmobiliario | Andes Capital Real Estate

## 1. Contexto del negocio

Andes Capital Real Estate desarrolla actividades comerciales en el sector inmobiliario mediante la venta de diferentes tipos de propiedades, atendiendo a distintos segmentos de compradores y utilizando diversos canales comerciales.

La información de estas operaciones se encuentra distribuida en registros transaccionales y tablas descriptivas de clientes y propiedades. Su análisis permite obtener una visión integral del desempeño comercial, identificar patrones de ventas y comprender qué factores tienen mayor participación en la generación de ingresos.

## 2. Objetivo del análisis

Evaluar el desempeño comercial de Andes Capital Real Estate durante 2023 y 2024, mediante el análisis integrado de ventas, clientes, propiedades y comportamiento temporal.

El análisis se orienta a identificar oportunidades de mejora en la estrategia comercial, comprender la contribución de los diferentes canales y segmentos de compradores, y estudiar la recurrencia de los clientes a través del tiempo.

## 3. Alcance del análisis

El proyecto aborda cuatro dimensiones principales:

- **Desempeño financiero y comercial:** ingresos generados, cantidad de ventas, ticket promedio y comisiones.
- **Composición comercial:** desempeño por tipo de propiedad, ciudad, canal de venta y segmento de comprador.
- **Evolución temporal:** tendencias de ventas, crecimiento interanual e indicadores acumulados.
- **Comportamiento de clientes:** análisis de cohortes para estudiar la recurrencia y las compras posteriores a la primera transacción.

## 4. Fuentes de información

El análisis utiliza tres conjuntos de datos principales:

- `hecho_ventas_propiedades`: registros de las transacciones comerciales.
- `dim_clientes`: características y segmentación de los compradores.
- `dim_propiedades`: atributos y características de los inmuebles.

Adicionalmente, se construye una dimensión temporal (`Dim_Fecha`) para facilitar el análisis histórico y las comparaciones entre periodos.


## 🧹 Paso 1: Limpieza de datos

Se realizó una revisión inicial de los conjuntos de datos en Power Query para verificar la consistencia de la información y asegurar que los campos utilizados en el análisis tuvieran los formatos adecuados.

### Validación de tipos de datos

Se revisaron y ajustaron los tipos de datos de las principales columnas:

- `fecha_venta`: se configuró como tipo fecha para permitir el análisis temporal.
- `precio_venta`, `monto_comision` y `precio_publicado`: se configuraron como valores decimales para realizar correctamente los cálculos monetarios.
- `porcentaje_comision`: se ajustó a formato porcentual.

### Configuración regional

Durante la preparación se revisó la interpretación de los valores decimales de `porcentaje_comision`. Para evitar errores de conversión, se utilizó la configuración regional English (United States), que permitió interpretar correctamente los valores con punto decimal.

### Validación de integridad

Se revisaron los valores nulos y los registros duplicados en los campos identificadores `id_venta`, `id_propiedad` e `id_cliente`.

Como resultado de estas comprobaciones, no se encontraron valores nulos ni registros duplicados en los campos evaluados.

Estas validaciones permitieron verificar la consistencia de los identificadores utilizados en el análisis y preparar la información para la construcción del modelo de datos.

## 📅 Paso 2: Creación de tabla calendario


Para facilitar el análisis temporal y las comparaciones entre periodos, se creó una dimensión de fechas denominada `Dim_Fecha` en Power BI.

Esta tabla permite organizar las transacciones por año, mes y trimestre, y sirve como base para calcular indicadores de inteligencia de tiempo, como ventas acumuladas (YTD), comparaciones interanuales (YoY) y variaciones mensuales (MoM).

### Construcción mediante DAX

La tabla se construyó utilizando las funciones `CALENDAR` y `ADDCOLUMNS`.

```dax
Dim_Fecha =
ADDCOLUMNS(
    CALENDAR(
        MIN(hecho_ventas_propiedades[fecha_venta]),
        MAX(hecho_ventas_propiedades[fecha_venta])
    ),
    "Año", YEAR([Date]),
    "Mes", FORMAT([Date], "MMMM"),
    "Mes Numero", MONTH([Date]),
    "Año-Mes", FORMAT([Date], "YYYY-MM"),
    "Trimestre", "T" & QUARTER([Date]),
    "Trimestre Numero", QUARTER([Date]),
    "Año Trimestre", YEAR([Date]) & "-T" & QUARTER([Date])
)
```

Se utilizó `CALENDAR` para generar un registro por cada fecha comprendida entre la fecha mínima y máxima de `fecha_venta`. Al obtener estos límites directamente de la tabla de ventas, el rango del calendario se adapta a la información disponible.

Posteriormente, `ADDCOLUMNS` permitió incorporar atributos temporales derivados de cada fecha, facilitando la agrupación y el análisis de las transacciones en diferentes niveles de tiempo.

### Campos de la dimensión temporal

La tabla incorpora los siguientes campos:

| Campo | Descripción | Utilidad |
|---|---|---|
| `Date` | Fecha completa | Relacionar el calendario con las transacciones |
| `Año` | Año correspondiente | Comparaciones anuales |
| `Mes` | Nombre del mes | Visualizaciones mensuales |
| `Mes Numero` | Número del mes | Orden cronológico de los meses |
| `Año-Mes` | Año y mes en formato YYYY-MM | Seguimiento mensual |
| `Trimestre` | Trimestre de la fecha | Análisis trimestral |
| `Trimestre Numero` | Número del trimestre | Orden cronológico de los trimestres |
| `Año Trimestre` | Año y trimestre | Comparaciones trimestrales |

### Orden cronológico de los periodos

Para garantizar que los periodos mensuales se presentaran en orden cronológico en las visualizaciones, se creó una columna calculada denominada `Año-Mes Orden`.

```dax
Año-Mes Orden =
YEAR(Dim_Fecha[Date]) * 100 + MONTH(Dim_Fecha[Date])
```

Esta columna combina el año y el número del mes en un valor numérico con formato YYYYMM.

Por ejemplo:

| Año-Mes | Año-Mes Orden |
|---|---:|
| 2023-01 | 202301 |
| 2023-02 | 202302 |

De esta manera, cada periodo cuenta con una clave de orden que permite mantener la secuencia temporal correcta y evitar que los meses se organicen alfabéticamente.

La columna `Año-Mes Orden` funciona como criterio de ordenamiento del campo `Año-Mes`, sin reemplazarlo en las visualizaciones.

### Integración al modelo de datos

La dimensión temporal se incorporó al modelo de datos y se relacionó con la tabla central `hecho_ventas_propiedades` mediante los campos `Dim_Fecha[Date]` y `hecho_ventas_propiedades[fecha_venta]`.

Se estableció una relación de uno a muchos (1:*) con dirección de filtrado simple, donde `Dim_Fecha` representa el lado uno y la tabla de ventas el lado muchos.

El rango generado comprende desde el 1 de enero de 2023 hasta el 31 de diciembre de 2024, correspondiente al periodo de las transacciones analizadas.

### Resultado y utilidad

La construcción de `Dim_Fecha` permitió establecer una dimensión temporal común para el análisis comercial y financiero.

Su integración al modelo facilitó:

- Analizar la evolución mensual y trimestral de los ingresos.
- Comparar el desempeño entre diferentes años.
- Calcular indicadores acumulados como YTD y MTD.
- Evaluar el crecimiento interanual (YoY).
- Mantener el orden cronológico de los periodos en las visualizaciones.

Esta estructura proporciona una base consistente para los análisis temporales desarrollados en el dashboard de Power BI..

## 🧩 Paso 3: Modelado de datos

Para organizar la información y facilitar el análisis comercial, se implementó un modelo de datos basado en un **esquema estrella (Star Schema)**. Esta estructura permite conectar las transacciones inmobiliarias con las dimensiones que aportan información adicional sobre clientes, propiedades y periodos de tiempo.

### Estructura del modelo

La tabla central del modelo es `hecho_ventas_propiedades`, que contiene los registros de las operaciones de venta.

Alrededor de esta tabla se integraron tres dimensiones:

- **`dim_clientes`:** contiene información y características de los clientes, incluida su segmentación.
- **`dim_propiedades`:** almacena las características de los inmuebles, como el tipo de propiedad y su ubicación.
- **`Dim_Fecha`:** permite organizar las transacciones por fechas, meses, trimestres y años.

Esta organización permite analizar las ventas desde diferentes perspectivas sin necesidad de relacionar directamente las tablas dimensionales entre sí.

### Configuración de las relaciones

Se establecieron relaciones entre la tabla de ventas y cada una de las dimensiones, utilizando las claves correspondientes.

La configuración del modelo contempla los siguientes elementos:

| Elemento | Configuración |
|---|---|
| Cardinalidad | Uno a muchos (1:*) |
| Dirección de filtro | Simple (Single) |
| Tabla del lado uno | Tablas dimensionales |
| Tabla del lado muchos | `hecho_ventas_propiedades` |

En el caso de la dimensión temporal, la relación se estableció entre `Dim_Fecha[Date]` y `hecho_ventas_propiedades[fecha_venta]`.

La dirección de filtrado permite que las dimensiones condicionen los registros de la tabla de ventas. De esta manera, al seleccionar un cliente, tipo de propiedad, ciudad o periodo, los indicadores y visualizaciones pueden actualizarse de acuerdo con el contexto seleccionado.

### Justificación del modelo

Se eligió el esquema estrella porque separa los registros de las transacciones de los atributos utilizados para analizarlas.

Esta estructura facilita la segmentación de los indicadores, simplifica la construcción de medidas DAX y permite mantener una organización clara de la información dentro de Power BI.

Además, evita establecer relaciones innecesarias entre las dimensiones, lo que contribuye a reducir posibles ambigüedades en la propagación de filtros.

**Resultado:** se implementó un modelo de datos estructurado que integra las ventas, los clientes, las propiedades y la información temporal, proporcionando la base para desarrollar los indicadores y análisis del dashboard.

## 📊 Paso 4: Creación de medidas y cálculos DAX

Una vez estructurado el modelo de datos, se desarrollaron diferentes medidas utilizando DAX (Data Analysis Expressions) en Power BI. El propósito fue transformar los registros de las transacciones inmobiliarias en indicadores que permitieran evaluar el desempeño financiero y comercial de Andes Capital Real Estate.

Las medidas se organizaron en diferentes grupos, de acuerdo con el tipo de análisis que permiten realizar.

### Medidas base

Se crearon cuatro medidas principales para obtener una visión general del desempeño del negocio: Ingreso Total, Cantidad de Ventas, Ticket Promedio y Comisión Total.

**Ingreso Total**

Calcula la suma de los valores registrados en la columna `precio_venta`, permitiendo conocer los ingresos generados por las operaciones inmobiliarias.

```dax
Ingreso Total =
SUM(hecho_ventas_propiedades[precio_venta])
```

**Cantidad de Ventas**

Cuenta el número de registros de operaciones inmobiliarias almacenados en la tabla `hecho_ventas_propiedades`.

```dax
Cantidad de Ventas =
COUNTROWS(hecho_ventas_propiedades)
```

**Ticket Promedio**

Calcula el valor promedio de las operaciones inmobiliarias a partir de los registros de la columna `precio_venta`.

```dax
Ticket Promedio =
AVERAGE(hecho_ventas_propiedades[precio_venta])
```

**Comisión Total**

Calcula la suma de los valores registrados en la columna `monto_comision`, permitiendo identificar el total de comisiones generadas por las operaciones.

```dax
Comisión Total =
SUM(hecho_ventas_propiedades[monto_comision])
```

Estas cuatro medidas se utilizaron principalmente en las tarjetas del Overview Ejecutivo, proporcionando una lectura rápida de los ingresos, el volumen de operaciones, el valor promedio de venta y las comisiones generadas.

---

### Medidas de participación porcentual

Se desarrollaron tres medidas para analizar la participación de los ingresos según el tipo de propiedad, el segmento de cliente y el canal de venta.

El objetivo de estos cálculos es identificar qué proporción del ingreso total representa cada categoría y facilitar la comparación de su contribución al desempeño comercial.

Para construir estas medidas se utilizaron las funciones `DIVIDE()`, `CALCULATE()` y `ALL()` de DAX, que permiten modificar el contexto de filtro y calcular la participación de cada categoría respecto al total correspondiente.

**1. Porcentaje de ingresos por tipo de propiedad**

Esta medida calcula qué porcentaje del ingreso total corresponde a cada tipo de propiedad. Para ello, elimina el filtro aplicado sobre la columna `tipo_propiedad` al calcular el denominador.

```dax
% Ingresos por Tipo de propiedad =
DIVIDE(
    [Ingreso Total],
    CALCULATE(
        [Ingreso Total],
        ALL(hecho_ventas_propiedades[tipo_propiedad])
    )
)
```

**2. Porcentaje de ingresos por segmento**

Permite identificar qué proporción de los ingresos corresponde a cada segmento de clientes. En este caso, se elimina el filtro aplicado sobre la columna `segmento_comprador` de la tabla `dim_clientes`.

```dax
% Ingresos por Segmento =
DIVIDE(
    [Ingreso Total],
    CALCULATE(
        [Ingreso Total],
        ALL(dim_clientes[segmento_comprador])
    )
)
```

**3. Porcentaje de ingresos por canal**

Esta medida permite conocer la participación de cada canal de venta en los ingresos totales. Para calcularla, se elimina el filtro aplicado sobre la columna `canal_venta`.

```dax
% Ingresos por Canal =
DIVIDE(
    [Ingreso Total],
    CALCULATE(
        [Ingreso Total],
        ALL(hecho_ventas_propiedades[canal_venta])
    )
)
```

Estas medidas permiten comparar la contribución de los diferentes tipos de propiedad, segmentos de clientes y canales de venta. Se utilizaron como indicadores complementarios en las visualizaciones del dashboard para facilitar la interpretación de la composición de los ingresos.

### Medidas de inteligencia de tiempo

Para analizar la evolución temporal de los ingresos, se desarrollaron medidas de inteligencia de tiempo utilizando la tabla `Dim_Fecha` y diferentes funciones DAX.

Estas medidas permiten comparar los resultados entre periodos, calcular ingresos acumulados y evaluar el crecimiento del negocio tanto a nivel mensual como anual.

Se implementaron las siguientes medidas:

**1. Ventas Mes Anterior**

Calcula los ingresos correspondientes al mes anterior al periodo seleccionado. Para ello, utiliza `CALCULATE()` y `DATEADD()` para desplazar el contexto de fechas un mes hacia atrás.

```dax
Ventas Mes Anterior =
CALCULATE(
    [Ingreso Total],
    DATEADD(Dim_Fecha[Date], -1, Month)
)
```

**2. Ventas Año Anterior**

Calcula los ingresos del mismo periodo del año anterior, desplazando el contexto de fechas doce meses hacia atrás mediante `DATEADD()`.

```dax
Ventas Año Anterior =
CALCULATE(
    [Ingreso Total],
    DATEADD(Dim_Fecha[Date], -1, YEAR)
)
```

**3. Ventas YTD (Year to Date)**

Calcula los ingresos acumulados desde el inicio del año hasta la fecha correspondiente al contexto de análisis. Para ello, utiliza `TOTALYTD()`.

```dax
Ventas YTD =
TOTALYTD(
    [Ingreso Total],
    Dim_Fecha[Date]
)
```

**4. Ventas YTD Año Anterior**

Permite obtener los ingresos acumulados del mismo periodo del año anterior. Utiliza `SAMEPERIODLASTYEAR()` para trasladar el contexto de fechas al año anterior y calcular el acumulado correspondiente.

```dax
Ventas YTD Año Anterior =
CALCULATE(
    [Ventas YTD],
    SAMEPERIODLASTYEAR(Dim_Fecha[Date])
)
```
**Consideración sobre el contexto de filtro de la medida YoY**

El resultado de la medida `YoY %` depende del periodo seleccionado en los filtros del dashboard. Cuando se selecciona únicamente 2024, la medida compara los ingresos de ese año con los de 2023 y muestra un crecimiento del 11,14 %.

Sin embargo, cuando se seleccionan simultáneamente 2023 y 2024, el cálculo considera los ingresos acumulados de ambos años y los compara con el periodo desplazado un año hacia atrás. Por esta razón, el resultado cambia a 111,14 %.

Este último valor no debe interpretarse como el crecimiento interanual de 2024. Para realizar esa interpretación, es necesario seleccionar exclusivamente 2024 y mantener la comparación con 2023.

**5. Ventas MTD (Month to Date)**

Calcula los ingresos acumulados desde el inicio del mes hasta la fecha correspondiente al contexto de análisis. Se utiliza la función `TOTALMTD()`.

```dax
Ventas MTD =
TOTALMTD(
    [Ingreso Total],
    Dim_Fecha[Date]
)
```

**6. Crecimiento interanual (YoY %)**

Calcula la variación porcentual de los ingresos frente al mismo periodo del año anterior. Compara el ingreso actual con el valor obtenido mediante la medida `Ventas Año Anterior`.

```dax
YoY % =
DIVIDE(
    [Ingreso Total] - [Ventas Año Anterior],
    [Ventas Año Anterior]
)
```

**7. Crecimiento mensual (MoM %)**

Calcula la variación porcentual de los ingresos respecto al mes anterior, permitiendo identificar el crecimiento o disminución entre ambos periodos.

```dax
MoM % =
DIVIDE(
    [Ingreso Total] - [Ventas Mes Anterior],
    [Ventas Mes Anterior]
)
```

**Lógica y utilidad de las medidas**

Las medidas desarrolladas permiten analizar el comportamiento de los ingresos desde diferentes perspectivas:

- **Comparación mensual:** identifica las variaciones de ingresos respecto al mes anterior mediante Ventas Mes Anterior y MoM %.
- **Comparación anual:** permite evaluar los resultados frente al mismo periodo del año anterior mediante Ventas Año Anterior y YoY %.
- **Acumulados anuales:** facilita el seguimiento de los ingresos generados desde el inicio del año mediante Ventas YTD.
- **Comparación de acumulados:** permite contrastar el desempeño acumulado del año actual con el periodo equivalente del año anterior mediante Ventas YTD Año Anterior.
- **Acumulados mensuales:** permite observar los ingresos generados desde el inicio del mes mediante Ventas MTD.

Estas medidas complementan los indicadores comerciales y permiten evaluar la evolución del negocio, identificar tendencias y comparar el desempeño entre diferentes periodos.

---

###  Medidas para análisis de cohortes

El análisis de cohortes busca identificar si los clientes realizan compras posteriores a su primera transacción y cómo evoluciona su comportamiento a lo largo del tiempo.

Para desarrollar este análisis se crearon cuatro columnas calculadas dentro de la tabla `hecho_ventas_propiedades`. Estas permiten identificar la fecha de adquisición de cada cliente, agruparlo según su mes y año de primera compra y calcular el tiempo transcurrido hasta cada transacción posterior.

**1. Primera compra**

Identifica la fecha de la primera transacción registrada para cada cliente. Para ello, se utiliza `MIN()` para obtener la fecha más antigua y `ALLEXCEPT()` para conservar el contexto del identificador del cliente.

```dax
Primera compra =
CALCULATE(
    MIN(hecho_ventas_propiedades[fecha_venta]),
    ALLEXCEPT(
        hecho_ventas_propiedades,
        hecho_ventas_propiedades[id_cliente]
    )
)
```

**2. Mes Cohorte**

Identifica el mes y año en que cada cliente realizó su primera compra. La función `FORMAT()` permite convertir la fecha en un formato de año y mes (`YYYY-MM`), facilitando la agrupación de los clientes por periodo de adquisición.

```dax
Mes Cohorte =
FORMAT([Primera compra], "YYYY-MM")
```

**3. Mes Venta**

Identifica la diferencia en meses calendario entre la fecha de primera compra del cliente y la fecha de cada transacción posterior. Para ello, se utiliza `DATEDIFF()` con el intervalo `MONTH`, que permite asignar cada compra a un mes relativo dentro del análisis de cohortes.

```dax
Mes Venta =
DATEDIFF(
    [Primera compra],
    hecho_ventas_propiedades[fecha_venta],
    MONTH
)
```
**4. Año Cohorte**

Identifica el año en que cada cliente realizó su primera compra.

```dax
Año Cohorte =
YEAR(hecho_ventas_propiedades[Primera compra])
```

Esta columna permite clasificar a los clientes según el año de adquisición, tomando como referencia la fecha de su primera compra.

Se incorporó para utilizarla como criterio de segmentación en la matriz de cohortes. De esta manera, es posible filtrar los clientes por el año en que fueron adquiridos sin limitar el análisis de sus compras posteriores.


**Resultado:** estas cuatro columnas permiten identificar la fecha y el periodo de adquisición de cada cliente, así como el tiempo transcurrido desde su primera compra hasta las transacciones posteriores. Constituyen la base para construir la matriz de cohortes y analizar el comportamiento de recompra.

### Medida de retención de clientes

Una vez creadas las columnas necesarias para identificar las cohortes, se desarrolló la medida `Retencion %` para analizar la proporción de clientes que vuelve a realizar compras en cada mes relativo a su primera transacción.

La medida utiliza `DISTINCTCOUNT()` para contar clientes únicos y `CALCULATE()` para establecer como referencia el número de clientes que realizó su primera compra (Mes Venta = 0).

**Retencion %**

```dax
Retencion % =
DIVIDE(
    DISTINCTCOUNT(hecho_ventas_propiedades[id_cliente]),
    CALCULATE(
        DISTINCTCOUNT(hecho_ventas_propiedades[id_cliente]),
        hecho_ventas_propiedades[Mes Venta] = 0
    )
)
```

**Interpretación de los resultados**

- **Mes 0:** representa el 100 % de los clientes de la cohorte, porque corresponde al periodo de su primera compra.
- **Mes 1:** muestra la proporción de clientes que realizó una compra durante el primer mes posterior a su adquisición.
- **Mes 2:** representa la proporción de clientes que compró durante el segundo mes posterior.
- **Meses siguientes:** permiten observar cómo evoluciona la actividad de compra de cada cohorte a lo largo del tiempo.

Esta medida se utilizó como valor de la matriz de cohortes, permitiendo comparar el comportamiento de recompra de los clientes según su mes de adquisición.

**Consideración sobre la interpretación de la medida**

Aunque la medida se denomina `Retencion %`, su cálculo representa la proporción de clientes de una cohorte que realizó una compra en un mes relativo específico, tomando como referencia el total de clientes de esa cohorte en el mes 0.

Por lo tanto, no mide una retención continua ni exige que el cliente haya realizado compras en todos los meses anteriores. Un cliente puede no registrar transacciones durante uno o varios meses y volver a comprar posteriormente, apareciendo nuevamente en el cálculo del mes relativo correspondiente.

### **Medida complementaria: Clientes con Compra**

Para complementar el análisis porcentual de la matriz de cohortes, creamos una medida que permite identificar la cantidad absoluta de clientes que realizaron compras en cada mes relativo.

```dax
Clientes con Compra =
DISTINCTCOUNT(hecho_ventas_propiedades[id_cliente])
```

Esta medida calcula el número de clientes únicos que realizaron una compra, respetando el contexto de filtros aplicado en la visualización.

Se utiliza en una página auxiliar de información sobre herramientas (*Tooltip*), permitiendo consultar la cantidad de clientes que representa el porcentaje mostrado en cada celda de la matriz.

De esta manera, el análisis combina dos perspectivas:

- **Retencion %:** muestra la proporción de clientes de una cohorte que realizó compras en un mes relativo determinado.
- **Clientes con Compra:** muestra la cantidad absoluta de clientes que realizaron compras en ese mismo periodo.
---

## 📊 Paso 5: Estructura del reporte

---

### Overview Ejecutivo

La página **Overview Ejecutivo** presenta una visión general del desempeño comercial y financiero de Andes Capital Real Estate durante el periodo 2023–2024. Su propósito es ofrecer una lectura rápida de los principales indicadores del negocio, identificar tendencias en los ingresos inmobiliarios y comparar los resultados de las dos ciudades analizadas.

**Indicadores principales (KPIs)**

En la parte superior del dashboard se incorporaron cinco indicadores:

- **Ingreso Total:** suma de los ingresos generados por las ventas inmobiliarias.
- **Cantidad de Ventas:** número total de transacciones registradas.
- **Ticket Promedio:** valor promedio de las ventas realizadas.
- **Comisión Total:** suma de las comisiones generadas por las transacciones.
- **Crecimiento Year over Year (YoY):** indicador de variación interanual de los ingresos.

Estos indicadores permiten evaluar de forma resumida el volumen de actividad comercial, el valor de las operaciones y el comportamiento de los ingresos.

**Evolución mensual de los ingresos**

Se incorporó un gráfico de líneas para visualizar el comportamiento mensual de los ingresos inmobiliarios. Esta visualización permite identificar variaciones entre meses, periodos de mayor facturación y fluctuaciones en el desempeño comercial.

El gráfico incluye una línea horizontal de referencia que facilita la comparación de los ingresos mensuales frente a un nivel común. También permite observar la evolución a lo largo de los dos años y detectar diferencias entre los periodos analizados.

**Comparación de ingresos por ciudad**

Se utilizó un gráfico de barras horizontales para comparar los ingresos inmobiliarios de Bogotá y Ciudad de México. Esta visualización permite identificar cuál de las dos ciudades concentra un mayor volumen de facturación.

**Cantidad de ventas por ciudad**

Un segundo gráfico de barras horizontales presenta el número de transacciones realizadas en cada ciudad. Su inclusión permite contrastar el volumen de operaciones con los ingresos generados y evitar que el análisis se limite únicamente a la facturación total.

**Filtros e interactividad**

La página incorpora filtros que permiten segmentar la información y explorar distintos escenarios:

- **Ciudad:** permite seleccionar una ciudad específica o consultar ambas.
- **Tipo de propiedad:** permite filtrar los resultados según la categoría del inmueble.
- **Fecha:** permite delimitar el periodo de análisis mediante un rango de fechas.
- **Año:** permite seleccionar uno o varios años para comparar los resultados.

La interacción entre estos filtros y las visualizaciones facilita el análisis de los indicadores bajo diferentes contextos temporales, geográficos y comerciales.

**Lectura general del dashboard**

Para el periodo completo 2023–2024, el dashboard registra ingresos por aproximadamente `$6.013 millones`, correspondientes a 8.500 ventas, con un ticket promedio cercano a `$707.353` y comisiones por aproximadamente $200,6 millones.

En la comparación por ciudades, Ciudad de México presenta mayores ingresos inmobiliarios, mientras que Bogotá registra una mayor cantidad de transacciones. Esta diferencia permite observar que un mayor volumen de ventas no necesariamente implica una mayor facturación total.

Al seleccionar únicamente 2024, los ingresos alcanzan aproximadamente $3.165 millones y se registran 4.455 ventas. En este contexto, el indicador YoY muestra un crecimiento del 11,14 % frente al año anterior.

---

### Análisis Comercial

La página **Análisis Comercial** profundiza en la composición de los ingresos inmobiliarios, permitiendo identificar las categorías de propiedades, los canales de venta y los segmentos de clientes que tienen mayor participación en los resultados del negocio.

El objetivo es comprender cómo se distribuyen las ventas y los ingresos, así como comparar el desempeño comercial entre las ciudades analizadas.

**Ingresos por tipo de propiedad**

Se incorporó un gráfico de columnas que compara los ingresos generados por cada categoría de propiedad: casas, inmuebles comerciales y departamentos.

Esta visualización permite identificar las categorías con mayor contribución a la facturación total. Durante el periodo analizado, las casas lideran los ingresos, con aproximadamente $2.241 millones, equivalentes al 37,26 % del total.

**Ingresos por canal de venta**

Un segundo gráfico compara los ingresos generados por los canales Corredor y Directo.

Los resultados muestran que el canal Corredor concentra la mayor parte de la facturación, con aproximadamente `$4.380` millones, mientras que el canal Directo registra cerca de `$1.633` millones. Esta diferencia evidencia el peso del canal de intermediación en la actividad comercial de la compañía.

**Ingresos por segmento de comprador**

El tercer gráfico presenta la distribución de los ingresos según los segmentos de clientes: Primera vez, Inversionista y Alto patrimonio.

El segmento Primera vez representa la mayor contribución, con aproximadamente `$3.800` millones. Le siguen los inversionistas, con cerca de `$1.500` millones, y el segmento de Alto patrimonio, con aproximadamente $800 millones.

Esta comparación permite reconocer qué segmentos tienen mayor participación en la generación de ingresos.

**Desempeño comercial por tipo de propiedad y ciudad**

En la parte inferior se incorporó una tabla que permite analizar conjuntamente el tipo de propiedad y la ciudad. La tabla presenta las siguientes métricas:

- **Cantidad de Ventas:** número de transacciones registradas para cada combinación de propiedad y ciudad.
- **Ticket Promedio:** valor promedio de las ventas realizadas en cada categoría y ciudad.
- **Ingreso Total:** facturación generada por cada combinación.
- **% Ingresos por Tipo de propiedad:** participación porcentual de los ingresos dentro del contexto de filtros aplicado.

La tabla utiliza formato condicional mediante una escala de colores para facilitar la identificación de diferencias entre los resultados de las categorías.

Este nivel de detalle permite comparar, por ejemplo, el comportamiento de las ventas de casas en Bogotá frente a Ciudad de México, así como identificar diferencias en el ticket promedio y los ingresos de cada tipo de inmueble.

**Filtros e interactividad**

La página incorpora tres filtros para segmentar el análisis:

- **Ciudad:** permite consultar los resultados de Bogotá, Ciudad de México o ambas.
- **Canal de venta:** permite comparar el desempeño de los canales Corredor y Directo.
- **Fecha:** permite delimitar el periodo de análisis mediante un rango de fechas.

Estos filtros actualizan las visualizaciones y la matriz, facilitando el análisis de la composición comercial bajo diferentes escenarios.

**Lectura general del dashboard**

Los resultados del periodo 2023–2024 muestran que las casas son la categoría con mayor contribución a los ingresos, mientras que el canal Corredor concentra ampliamente la facturación frente al canal Directo.

Por otra parte, los compradores de Primera vez representan el segmento con mayor generación de ingresos. En el análisis por ciudad y tipo de propiedad, la matriz permite observar diferencias tanto en el volumen de transacciones como en el ticket promedio.

En conjunto, esta página permite identificar los principales componentes de la actividad comercial y proporciona una base para evaluar oportunidades de crecimiento por categoría de propiedad, canal de venta, segmento de comprador y ubicación geográfica.

### Análisis de Cohortes

La página **Análisis de Cohortes** tiene como objetivo estudiar el comportamiento de recompra de los clientes a partir de su primera compra, permitiendo observar cómo evoluciona su actividad comercial durante los meses posteriores.

Para este análisis, los clientes se agruparon en cohortes mensuales según el mes en el que realizaron su primera compra. Posteriormente, se evaluó su actividad en los meses siguientes, utilizando como referencia el tiempo transcurrido desde esa primera transacción.

**Matriz de análisis de cohortes**

La visualización principal es una matriz de cohortes que organiza la información de la siguiente manera:

- **Mes Cohorte:** identifica el mes y año en que los clientes realizaron su primera compra. Cada fila representa una cohorte.
- **Mes relativo:** las columnas numeradas del 0 al 23 representan los meses calendario relativos a la primera compra de cada cliente. El mes 0 corresponde al mes de adquisición y los valores posteriores identifican los meses calendario en los que se registraron nuevas transacciones.
- **Retención %:** muestra la proporción de clientes de cada cohorte que realizó una compra durante el mes relativo correspondiente.

El mes 0 representa el periodo de la primera compra y tiene un valor del 100 %, ya que todos los clientes incluidos en la cohorte realizaron su primera transacción durante ese mes.

Los meses posteriores permiten observar qué proporción de esos clientes volvió a realizar compras en cada periodo relativo.

La tabla incorpora formato condicional mediante una escala de colores que facilita la identificación de variaciones en los porcentajes y permite comparar visualmente el comportamiento de las diferentes cohortes.

**Segmentación por año de cohorte**

Se incorporó un segmentador basado en la columna `Año Cohorte`, que permite filtrar a los clientes según el año en que realizaron su primera compra.

Este filtro permite analizar de manera independiente las cohortes de 2023 y 2024, manteniendo dentro del análisis las compras posteriores realizadas por los clientes de cada cohorte.

De esta manera, es posible estudiar el comportamiento de recompra de los clientes a lo largo del tiempo, considerando su año de adquisición.

**Información sobre herramientas: clientes con compra**

Se configuró una página auxiliar denominada `Tooltip – Clientes con Compra`, diseñada como información sobre herramientas de la matriz de cohortes.

Esta página contiene una tarjeta que utiliza la medida `Clientes con Compra`, la cual muestra la cantidad absoluta de clientes que realizaron compras en el periodo correspondiente.

Al pasar el cursor sobre una celda de la matriz, se presenta esta cantidad como información complementaria al porcentaje de recompra (`Retencion %`).

De esta manera, el análisis integra dos perspectivas: la proporción de clientes de cada cohorte que realizó compras en un mes relativo y la cantidad absoluta de clientes que efectuó dichas compras.

**Lectura e interpretación del análisis**

La matriz presenta los porcentajes de recompra correspondientes a cada cohorte y mes relativo, sin mostrar totales generales de filas ni de columnas.

Esta configuración permite concentrar la interpretación en los resultados individuales de cada cohorte, evitando porcentajes agregados que podrían resultar poco representativos, especialmente cuando los periodos de seguimiento son diferentes.

La lectura de los porcentajes se complementa con la información sobre herramientas, que muestra la cantidad absoluta de clientes que realizaron compras en cada periodo.

En términos generales, los resultados permiten observar variaciones en la actividad de compra entre cohortes y a lo largo de los meses posteriores a la primera transacción.

Es importante señalar que la medida utilizada representa la proporción de clientes que realizó una compra en cada mes relativo, y no una retención continua. Esto significa que un cliente puede no registrar compras durante uno o varios meses y volver a comprar posteriormente.

Asimismo, las cohortes de 2023 cuentan con una mayor cantidad de meses observados que las cohortes de 2024. Por esta razón, las comparaciones deben realizarse considerando ventanas de seguimiento equivalentes.

## 📝Paso 6:  Resumen Ejecutivo


###  Hallazgos clave

A partir del análisis realizado en Power BI, se identificaron cinco hallazgos principales sobre el desempeño comercial de Andes Capital Real Estate durante el periodo 2023–2024:

1. **Desempeño financiero:** la empresa registró ingresos totales por aproximadamente $6.013 millones , correspondientes a 8.500 ventas, con un ticket promedio de `$707.353` y comisiones por aproximadamente `$200,6` millones.

2. **Ingresos por tipo de propiedad:** las casas generaron los mayores ingresos, con aproximadamente $2.241 millones. Sin embargo, los departamentos concentraron el mayor volumen de transacciones, con 5.105 ventas. Esto evidencia que la categoría con más operaciones no necesariamente es la que genera mayor facturación.

3. **Diferencias entre ciudades:** Bogotá registró 4.377 ventas, superando las 4.123 de Ciudad de México. No obstante, Ciudad de México generó mayores ingresos, con aproximadamente `$3.242` millones frente a `$2.770` millones de Bogotá.

4. **Desempeño de los canales comerciales:** el canal Corredor concentró aproximadamente el 72,85 % de los ingresos, convirtiéndose en el principal canal de generación de facturación para la empresa.

5. **Composición de los segmentos de clientes:** el segmento Primera vez representó la mayor participación de ingresos, con un 62,93 %, seguido por Inversionista, con 24,47 %, y Alto patrimonio, con 12,60 %.

En conjunto, estos hallazgos permiten identificar las principales fuentes de ingresos, las diferencias entre mercados y el peso de cada categoría comercial en los resultados de la empresa.

### Métricas principales

Para consolidar los resultados del análisis, se presentan los principales indicadores financieros y comerciales de Andes Capital Real Estate durante el periodo 2023–2024.

| Indicador | Resultado |
|---|---:|
| Ingreso Total | \$6.012.502.170 |
| Cantidad de Ventas | 8.500 |
| Ticket Promedio | \$707.353,20 |
| Comisión Total | \$200.627.166 |
| Crecimiento YoY | 11,14 % |

Estos indicadores permiten resumir el desempeño del negocio desde cinco perspectivas:

- **Ingreso Total:** representa el valor acumulado de las transacciones inmobiliarias.
- **Cantidad de Ventas:** refleja el volumen de operaciones realizadas.
- **Ticket Promedio:** muestra el valor promedio de las transacciones.
- **Comisión Total:** representa el monto acumulado de las comisiones generadas.
- **Crecimiento YoY:** evidencia la variación porcentual de los ingresos de 2024 frente a 2023.

En conjunto, los indicadores muestran que la empresa registró un crecimiento interanual positivo en sus ingresos, acompañado de un volumen de 8.500 transacciones durante los dos años analizados.

### Insights accionables

A partir de los resultados obtenidos, se identificaron tres insights que permiten comprender mejor el comportamiento comercial de Andes Capital Real Estate:

**1. Alta participación de compradores de primera vez**

El segmento Primera vez concentra el 62,93 % de los ingresos, lo que evidencia su importancia dentro de la generación de facturación. Este resultado plantea la oportunidad de fortalecer las estrategias de seguimiento y recompra para convertir a los nuevos compradores en clientes recurrentes.

**2. Comportamiento de recompra de clientes**

El análisis de cohortes permite observar la proporción de clientes que realiza compras en cada mes relativo a su primera compra. El dashboard incorpora un filtro por año de cohorte y una medida complementaria que muestra la cantidad absoluta de clientes mediante información sobre herramientas, facilitando una interpretación más completa de los resultados.

Las cohortes de 2023 disponen de más meses de seguimiento que las de 2024. Por tanto, no se puede concluir que un año presente mayor recurrencia sin comparar periodos equivalentes de observación.

**3. Crecimiento de los ingresos**

Los ingresos de 2024 aumentaron un 11,14 % frente a 2023. Este resultado refleja una evolución positiva de la facturación y permite tomar 2024 como referencia para evaluar el desempeño comercial en periodos posteriores.

En conjunto, estos insights permiten identificar oportunidades relacionadas con la recompra de clientes, el seguimiento de cohortes y la evolución de los ingresos.

### Recomendaciones estratégicas

A partir de los hallazgos e insights identificados, se proponen cinco recomendaciones estratégicas orientadas a fortalecer el desempeño comercial de Andes Capital Real Estate:

**1. Priorizar la comercialización de casas**

Evaluar oportunidades para fortalecer la promoción y comercialización de casas, dado que esta categoría genera los mayores ingresos. Antes de incrementar su participación en la oferta, se recomienda contrastar estos resultados con los márgenes y las comisiones para determinar su contribución real a la rentabilidad.

**2. Fortalecer el canal de corredores**

Analizar las estrategias comerciales que explican la participación del canal Corredor, responsable del 72,85 % de los ingresos. Se recomienda identificar las prácticas que generan mejores resultados y evaluar cómo fortalecer este canal, sin asumir que una mayor facturación representa necesariamente una mayor rentabilidad.

**3. Desarrollar estrategias de recompra y fidelización**

Diseñar acciones de seguimiento para los clientes que realizan su primera compra, con el propósito de fomentar futuras transacciones. El análisis de cohortes puede utilizarse como herramienta de seguimiento para evaluar la evolución de la actividad de compra de estos clientes.

**4. Investigar las diferencias de desempeño entre ciudades**

Profundizar en las diferencias entre Bogotá y Ciudad de México, especialmente en el valor promedio de las operaciones, los tipos de propiedad comercializados y los segmentos de compradores. Esto permitiría identificar los factores que explican por qué Ciudad de México genera mayores ingresos a pesar de registrar menos transacciones.

**5. Mejorar el valor promedio de las ventas de departamentos**

Analizar oportunidades para incrementar el ticket promedio de los departamentos, considerando que concentran el mayor volumen de transacciones, pero no lideran la generación de ingresos. Las estrategias comerciales deberían buscar aumentar el valor de las operaciones sin comprometer el volumen de ventas.
